B"H

# Milestone 2: Cleaning and Formatting the Flat File Source

- DSC-540
- David Koyrakh
- Professor Catie Williams

## Load and examine the data

In [1]:
# Import necessary libraries
import pandas as pd

In [2]:
# Load the raw USDA Economic Research Service dataset
dataset = pd.read_csv("USDA_dataset.csv")

# Display head
print(dataset.head(100))

# Display the dimensions of the dataset
print("Dataset dimensions:", dataset.shape)

    FIPS_Code State      Area_Name                  Attribute        Value
0           0    US  United States  Civilian_labor_force_2000  142601576.0
1           0    US  United States              Employed_2000  136904853.0
2           0    US  United States            Unemployed_2000    5696723.0
3           0    US  United States     Unemployment_rate_2000          4.0
4           0    US  United States  Civilian_labor_force_2001  143786537.0
..        ...   ...            ...                        ...          ...
95       1000    AL        Alabama            Unemployed_2000      99442.0
96       1000    AL        Alabama     Unemployment_rate_2000          4.6
97       1000    AL        Alabama  Civilian_labor_force_2001    2128027.0
98       1000    AL        Alabama              Employed_2001    2017467.0
99       1000    AL        Alabama            Unemployed_2001     110560.0

[100 rows x 5 columns]
Dataset dimensions: (316633, 5)


By observing the dataset head and checking the file manually, it appears that the first ~100 rows are overall statistics on the whole United States, followed by over 316,500 rows with statistics on specific counties. This is organized by state, in alphabetic order.

### Ensure that all states are represented

In [3]:
# Count unique values for the columns: State, Area_Name, and Attribute
print("Count of unique 'State' values:", dataset['State'].nunique())
print("Count of unique 'Area_Name' values:", dataset['Area_Name'].nunique())
print("Count of unique 'Attribute' values:", dataset['Attribute'].nunique())

Count of unique 'State' values: 53
Count of unique 'Area_Name' values: 3276
Count of unique 'Attribute' values: 97


In [4]:
print("All unique 'State' values:\n", dataset['State'].unique())

All unique 'State' values:
 ['US' 'AL' 'AK' 'AZ' 'AR' 'CA' 'CO' 'CT' 'DE' 'DC' 'FL' 'GA' 'HI' 'ID'
 'IL' 'IN' 'IA' 'KS' 'KY' 'LA' 'ME' 'MD' 'MA' 'MI' 'MN' 'MS' 'MO' 'MT'
 'NE' 'NV' 'NH' 'NJ' 'NM' 'NY' 'NC' 'ND' 'OH' 'OK' 'OR' 'PA' 'RI' 'SC'
 'SD' 'TN' 'TX' 'UT' 'VT' 'VA' 'WA' 'WV' 'WI' 'WY' 'PR']


The dataset includes:
- 50 states (AL through WY)
- 1 federal district (DC)
- 1 territory (PR)
- 1 national aggregate (US)

This explains why there are 3 additional "State" variables above the expected count of 50 states.

## Step 1: Drop county-level data

Since this project is state-focused, data for each specific county is out of scope.

Thankfully, the dataset already comes with aggregate data for each state, so we will extract those out by removing county-specific data.

We will keep US/country-wide aggregate data, for potential use as a baseline for future comparisons. We will also keep DC and Puerto Rico (PR), for now.

### Check for county identifiers

In [5]:
# Display the first 150 rows excluding rows where State is 'US'
dataset_filtered = dataset[dataset['State'] != 'US']
print(dataset_filtered.head(150))

     FIPS_Code State           Area_Name                  Attribute      Value
93        1000    AL             Alabama  Civilian_labor_force_2000  2147173.0
94        1000    AL             Alabama              Employed_2000  2047731.0
95        1000    AL             Alabama            Unemployed_2000    99442.0
96        1000    AL             Alabama     Unemployment_rate_2000        4.6
97        1000    AL             Alabama  Civilian_labor_force_2001  2128027.0
..         ...   ...                 ...                        ...        ...
238       1001    AL  Autauga County, AL  Civilian_labor_force_2012    25762.0
239       1001    AL  Autauga County, AL              Employed_2012    23932.0
240       1001    AL  Autauga County, AL            Unemployed_2012     1830.0
241       1001    AL  Autauga County, AL     Unemployment_rate_2012        7.1
242       1001    AL  Autauga County, AL  Civilian_labor_force_2013    25783.0

[150 rows x 5 columns]


Based on the above, county-specific rows are unique in that they contain commas in the `Area_Name` column. Therefore, we can filter them out by targeting rows which match that criteria.

### Remove county-level rows from the dataset

In [6]:
# Count data before filtering
print("Count of unique 'Area_Name' values BEFORE filtering county-level data:", dataset['Area_Name'].nunique())

# Filter out the county-level data, using presence of a comma in the 'Area_Name' column as a marker
dataset = dataset[~dataset['Area_Name'].str.contains(',')]

# Re-count unique values
print("Count of unique `Area_Name` values AFTER filtering county-level data:", dataset['Area_Name'].nunique())

Count of unique 'Area_Name' values BEFORE filtering county-level data: 3276
Count of unique `Area_Name` values AFTER filtering county-level data: 53


In [7]:
# Display the first 150 rows excluding rows where State is 'US'
dataset_filtered = dataset[dataset['State'] != 'US']
print(dataset_filtered.head(150))

      FIPS_Code State Area_Name                  Attribute      Value
93         1000    AL   Alabama  Civilian_labor_force_2000  2147173.0
94         1000    AL   Alabama              Employed_2000  2047731.0
95         1000    AL   Alabama            Unemployed_2000    99442.0
96         1000    AL   Alabama     Unemployment_rate_2000        4.6
97         1000    AL   Alabama  Civilian_labor_force_2001  2128027.0
...         ...   ...       ...                        ...        ...
6737       2000    AK    Alaska     Unemployment_rate_2012        7.2
6738       2000    AK    Alaska  Civilian_labor_force_2013   363544.0
6739       2000    AK    Alaska              Employed_2013   338104.0
6740       2000    AK    Alaska            Unemployed_2013    25440.0
6741       2000    AK    Alaska     Unemployment_rate_2013        7.0

[150 rows x 5 columns]


All of the rows containing county-specific data have been removed.

## Step 2: Drop unnecessary columns

In [8]:
# List all column names in the dataset
print(dataset.columns)

Index(['FIPS_Code', 'State', 'Area_Name', 'Attribute', 'Value'], dtype='object')


We don't need the `FIPS_Code` column, so let's drop it.

In [9]:
dataset = dataset.drop(columns=['FIPS_Code'])
print(dataset.columns)

Index(['State', 'Area_Name', 'Attribute', 'Value'], dtype='object')


## Step 3: Make `Attribute` values more readable

The 'Attribute' column contains categorical names for sets of important data points, but isn't very readable. In this step, I will perform the following transformations to make the values more human-readable:
- Replace underscores with spaces
- Wrap the year in parentheses

In [10]:
import re

# Function to reformat Attribute names
def format_attribute(attribute):
    # Replace underscores with spaces
    attribute = attribute.replace("_", " ")
    # Move the year (if present) to parentheses at the end
    attribute = re.sub(r'(\d{4})$', r'(\1)', attribute)
    return attribute

# Apply formatting to the 'Attribute' column
dataset['Attribute'] = dataset['Attribute'].apply(format_attribute)

# Display the transformed dataset
dataset.head()

,State,Area_Name,Attribute,Value
0,US,United States,Civilian labor force (2000),142601576.0
1,US,United States,Employed (2000),136904853.0
2,US,United States,Unemployed (2000),5696723.0
3,US,United States,Unemployment rate (2000),4.0
4,US,United States,Civilian labor force (2001),143786537.0


The `Attribute` values are now more human-readable.

## Step 4: Handle duplicate entries

In order to pivot the dataset in the following step, we have to make sure that there are no duplicate entries for 'State', 'Area_Name', and 'Attribute' combinations. We will first check if there are, and if found, handle them accordingly.

In [11]:
# Identify rows that have duplicates for the combination (State, Area_Name, Attribute)
dupes_mask = dataset.duplicated(subset=['State', 'Area_Name', 'Attribute'], keep=False)

# Display results
if dupes_mask.any():
    print("Duplicates found:")
    print(dataset[dupes_mask].sort_values(['State', 'Area_Name', 'Attribute']))
else:
    print("No duplicates found.")

Duplicates found:
      State             Area_Name                    Attribute     Value
31727    DC  District of Columbia  Civilian labor force (2000)  309977.0
31824    DC  District of Columbia  Civilian labor force (2000)  309977.0
31731    DC  District of Columbia  Civilian labor force (2001)  311232.0
31828    DC  District of Columbia  Civilian labor force (2001)  311232.0
31735    DC  District of Columbia  Civilian labor force (2002)  308760.0
...     ...                   ...                          ...       ...
31907    DC  District of Columbia     Unemployment rate (2020)       7.9
31814    DC  District of Columbia     Unemployment rate (2021)       6.8
31911    DC  District of Columbia     Unemployment rate (2021)       6.8
31818    DC  District of Columbia     Unemployment rate (2022)       4.7
31915    DC  District of Columbia     Unemployment rate (2022)       4.7

[188 rows x 4 columns]


There are duplicates! Let's check if each duplicate has the same Value:

In [12]:
dupes = dataset[dataset.duplicated(subset=['State', 'Area_Name', 'Attribute'], keep=False)]

# Group duplicates, and count how many unique Value entires each group has
unique_value_counts = dupes.groupby(['State', 'Area_Name', 'Attribute'])['Value'].nunique()

# Identify which groups have more than 1 unique Value
duplicates_with_different_values = unique_value_counts[unique_value_counts > 1]
duplicates_with_different_values

Series([], Name: Value, dtype: int64)

All of the duplicates are identical, so duplicates can be easily dropped.

In [13]:
# Drop the duplicates
dataset = dataset.drop_duplicates(subset=['State', 'Area_Name', 'Attribute'])

## Step 5: Pivot attributes into columns (long to wide format transformation)

Currently, this dataset contains a large number of rows for each state, since each state has its own row for each individual `Attribute` field. The data will be much more readable if this row redundancy is reduced, by pivoting the `Attribute` fields into individual columns. This will result in a cleaner dataset - one state per row - and it will be more straightforward to read and process in the future.

In [14]:
dataset_pivoted = dataset.pivot(
    index=['State', 'Area_Name'], 
    columns='Attribute', 
    values='Value'
)
dataset_pivoted.reset_index(inplace=True)
dataset_pivoted

Attribute,State,Area_Name,Civilian labor force (2000),Civilian labor force (2001),Civilian labor force (2002),Civilian labor force (2003),Civilian labor force (2004),Civilian labor force (2005),Civilian labor force (2006),Civilian labor force (2007),...,Unemployment rate (2014),Unemployment rate (2015),Unemployment rate (2016),Unemployment rate (2017),Unemployment rate (2018),Unemployment rate (2019),Unemployment rate (2020),Unemployment rate (2021),Unemployment rate (2022),Urban Influence Code (2013)
0,AK,Alaska,319777.0,321652.0,327054.0,332494.0,337469.0,344157.0,348756.0,350233.0,...,6.7,6.3,6.6,6.5,6.0,5.6,8.3,6.4,4.0,NaN
1,AL,Alabama,2147173.0,2128027.0,2112621.0,2128668.0,2138306.0,2140356.0,2170007.0,2180448.0,...,6.7,6.1,5.9,4.5,3.9,3.2,6.4,3.4,2.6,NaN
2,AR,Arkansas,1260507.0,1261419.0,1276964.0,1282578.0,1302378.0,1333880.0,1346532.0,1350811.0,...,5.9,5.0,4.0,3.7,3.7,3.5,6.2,4.1,3.3,NaN
3,AZ,Arizona,2510611.0,2593664.0,2683502.0,2728649.0,2788618.0,2862517.0,2970679.0,3018805.0,...,6.8,6.1,5.5,5.0,4.8,4.8,7.8,5.1,3.8,NaN
4,CA,California,16837536.0,17096428.0,17246528.0,17271899.0,17374506.0,17537931.0,17661179.0,17910726.0,...,7.6,6.3,5.5,4.8,4.2,4.1,10.1,7.3,4.2,NaN
5,CO,Colorado,2358371.0,2396012.0,2448903.0,2490397.0,2539389.0,2572382.0,2640452.0,2687742.0,...,5.0,3.7,3.1,2.6,3.0,2.7,6.8,5.4,3.0,NaN
6,CT,Connecticut,1760697.0,1749195.0,1776064.0,1785737.0,1777510.0,1794301.0,1831083.0,1856008.0,...,6.6,5.6,4.8,4.4,3.9,3.6,7.9,6.3,4.2,NaN
7,DC,District of Columbia,309977.0,311232.0,308760.0,308454.0,313735.0,318130.0,320581.0,326557.0,...,7.7,6.9,6.2,6.1,5.7,5.5,7.9,6.8,4.7,1.0
8,DE,Delaware,412956.0,421563.0,417207.0,415890.0,421093.0,433375.0,443228.0,445306.0,...,5.6,4.8,4.5,4.5,3.7,3.6,7.5,5.5,4.5,NaN
9,FL,Florida,7917857.0,8048882.0,8050051.0,8123135.0,8347316.0,8647279.0,8927098.0,9113604.0,...,6.4,5.5,4.9,4.3,3.6,3.2,8.1,4.6,2.9,NaN


To ensure the new, pivoted dataset is clean and readable, we will perform a small sub-step here:

The Area_Name column now just represents state, so we can rename it accordingly.

In [15]:
# Rename the `Area_Name` column
dataset_pivoted = dataset_pivoted.rename(columns={'Area_Name': 'State (long)'})

# Display list of updated columns
list(dataset_pivoted.columns.values)

['State',
 'State (long)',
 'Civilian labor force (2000)',
 'Civilian labor force (2001)',
 'Civilian labor force (2002)',
 'Civilian labor force (2003)',
 'Civilian labor force (2004)',
 'Civilian labor force (2005)',
 'Civilian labor force (2006)',
 'Civilian labor force (2007)',
 'Civilian labor force (2008)',
 'Civilian labor force (2009)',
 'Civilian labor force (2010)',
 'Civilian labor force (2011)',
 'Civilian labor force (2012)',
 'Civilian labor force (2013)',
 'Civilian labor force (2014)',
 'Civilian labor force (2015)',
 'Civilian labor force (2016)',
 'Civilian labor force (2017)',
 'Civilian labor force (2018)',
 'Civilian labor force (2019)',
 'Civilian labor force (2020)',
 'Civilian labor force (2021)',
 'Civilian labor force (2022)',
 'Employed (2000)',
 'Employed (2001)',
 'Employed (2002)',
 'Employed (2003)',
 'Employed (2004)',
 'Employed (2005)',
 'Employed (2006)',
 'Employed (2007)',
 'Employed (2008)',
 'Employed (2009)',
 'Employed (2010)',
 'Employed (2011)

In [16]:
# Count number of rows in the pivoted dataset
len(dataset_pivoted)

53

Final preview of the cleaned dataset:

In [17]:
dataset_pivoted

Attribute,State,State (long),Civilian labor force (2000),Civilian labor force (2001),Civilian labor force (2002),Civilian labor force (2003),Civilian labor force (2004),Civilian labor force (2005),Civilian labor force (2006),Civilian labor force (2007),...,Unemployment rate (2014),Unemployment rate (2015),Unemployment rate (2016),Unemployment rate (2017),Unemployment rate (2018),Unemployment rate (2019),Unemployment rate (2020),Unemployment rate (2021),Unemployment rate (2022),Urban Influence Code (2013)
0,AK,Alaska,319777.0,321652.0,327054.0,332494.0,337469.0,344157.0,348756.0,350233.0,...,6.7,6.3,6.6,6.5,6.0,5.6,8.3,6.4,4.0,NaN
1,AL,Alabama,2147173.0,2128027.0,2112621.0,2128668.0,2138306.0,2140356.0,2170007.0,2180448.0,...,6.7,6.1,5.9,4.5,3.9,3.2,6.4,3.4,2.6,NaN
2,AR,Arkansas,1260507.0,1261419.0,1276964.0,1282578.0,1302378.0,1333880.0,1346532.0,1350811.0,...,5.9,5.0,4.0,3.7,3.7,3.5,6.2,4.1,3.3,NaN
3,AZ,Arizona,2510611.0,2593664.0,2683502.0,2728649.0,2788618.0,2862517.0,2970679.0,3018805.0,...,6.8,6.1,5.5,5.0,4.8,4.8,7.8,5.1,3.8,NaN
4,CA,California,16837536.0,17096428.0,17246528.0,17271899.0,17374506.0,17537931.0,17661179.0,17910726.0,...,7.6,6.3,5.5,4.8,4.2,4.1,10.1,7.3,4.2,NaN
5,CO,Colorado,2358371.0,2396012.0,2448903.0,2490397.0,2539389.0,2572382.0,2640452.0,2687742.0,...,5.0,3.7,3.1,2.6,3.0,2.7,6.8,5.4,3.0,NaN
6,CT,Connecticut,1760697.0,1749195.0,1776064.0,1785737.0,1777510.0,1794301.0,1831083.0,1856008.0,...,6.6,5.6,4.8,4.4,3.9,3.6,7.9,6.3,4.2,NaN
7,DC,District of Columbia,309977.0,311232.0,308760.0,308454.0,313735.0,318130.0,320581.0,326557.0,...,7.7,6.9,6.2,6.1,5.7,5.5,7.9,6.8,4.7,1.0
8,DE,Delaware,412956.0,421563.0,417207.0,415890.0,421093.0,433375.0,443228.0,445306.0,...,5.6,4.8,4.5,4.5,3.7,3.6,7.5,5.5,4.5,NaN
9,FL,Florida,7917857.0,8048882.0,8050051.0,8123135.0,8347316.0,8647279.0,8927098.0,9113604.0,...,6.4,5.5,4.9,4.3,3.6,3.2,8.1,4.6,2.9,NaN


We now have a clean, pivoted dataset to work with.

## Conclusion

The transformations in this notebook included:

1. Dropping county-level data to focus on state-level insights
2. Removing columns deemed unnecessary
3. Refining statistic/attribute labels for clarity
4. Eliminating duplicate entries
5. Pivoting the dataset into a wide format for easier analysis

The dataset comes from the United States Department of Agriculture's (USDA) Economic Research Service. Since it is completely free and public domain, it is not subject to much regulation. However, it should still be utilized and presented honestly and with good will. By dropping county-specific data, we lost granularity, which can lead to overlooking local nuances and misrepresenting regional differences. I assumed that the dataset did not contain any bad data, since it was government-sourced, but practically I should check for missing values, abnormal outliers, and values of incorrect or unexpected types. The data was obtained legally via download from the USDA's public website here: https://www.ers.usda.gov/data-products/county-level-data-sets/county-level-data-sets-download-data

In order to mitigate the risk of reduced granularity from my removal of county-level data, I plan to be transparent about performing that transformation. If appropriate, I may even write a disclaimer warning users of my prepared data from over-generalizing, and to keep in mind that state and national-level data misses many nuances of specific cities, towns, and sub-regions.

In [18]:
import pandas as pd
import numpy as np

# Save to parquet
dataset_pivoted.to_parquet('labor_force_data.parquet')

# Read it back
df_loaded = pd.read_parquet('labor_force_data.parquet')

# Compare the two DataFrames
def verify_dataframes(original, loaded):
    # Check if shapes match
    if original.shape != loaded.shape:
        print(f"Shape mismatch: Original {original.shape} vs Loaded {loaded.shape}")
        return False
        
    # Check column names and order
    if not original.columns.equals(loaded.columns):
        print("Column names or order don't match")
        print("Missing in loaded:", set(original.columns) - set(loaded.columns))
        print("Extra in loaded:", set(loaded.columns) - set(original.columns))
        return False
    
    # Check dtypes
    if not original.dtypes.equals(loaded.dtypes):
        print("Data types don't match")
        dtype_diff = pd.DataFrame({
            'original': original.dtypes,
            'loaded': loaded.dtypes
        }).loc[original.dtypes != loaded.dtypes]
        print(dtype_diff)
        
    # Check actual data values
    is_equal = original.equals(loaded)
    if not is_equal:
        # Find differences if not equal
        # For numeric columns, find where absolute difference exceeds a small threshold
        for col in original.select_dtypes(include=np.number).columns:
            mask = ~(original[col].fillna(0) - loaded[col].fillna(0)).abs().lt(1e-10)
            if mask.any():
                diff_indices = mask[mask].index
                print(f"Differences in column '{col}' at indices: {diff_indices.tolist()}")
                print("Sample differences:")
                for idx in diff_indices[:5]:  # Show first 5 differences
                    print(f"  Index {idx}: Original={original.loc[idx, col]}, Loaded={loaded.loc[idx, col]}")
        
        # For non-numeric columns, find direct mismatches
        for col in original.select_dtypes(exclude=np.number).columns:
            mask = original[col] != loaded[col]
            if mask.any():
                diff_indices = mask[mask].index
                print(f"Differences in column '{col}' at indices: {diff_indices.tolist()}")
                print("Sample differences:")
                for idx in diff_indices[:5]:  # Show first 5 differences
                    print(f"  Index {idx}: Original={original.loc[idx, col]}, Loaded={loaded.loc[idx, col]}")
    
    return is_equal

# Run the verification
is_identical = verify_dataframes(dataset_pivoted, df_loaded)
print("Verification result:", "Success - DataFrames are identical" if is_identical else "Failed - DataFrames differ")

Verification result: Success - DataFrames are identical


In [27]:
if dataset_pivoted.equals(df_loaded):
    print("Verification passed - DataFrames are identical")
else:
    print("Verification failed - DataFrames differ")

Verification passed - DataFrames are identical


In [19]:
dataset_pivoted.head()

Attribute,State,State (long),Civilian labor force (2000),Civilian labor force (2001),Civilian labor force (2002),Civilian labor force (2003),Civilian labor force (2004),Civilian labor force (2005),Civilian labor force (2006),Civilian labor force (2007),...,Unemployment rate (2014),Unemployment rate (2015),Unemployment rate (2016),Unemployment rate (2017),Unemployment rate (2018),Unemployment rate (2019),Unemployment rate (2020),Unemployment rate (2021),Unemployment rate (2022),Urban Influence Code (2013)
0,AK,Alaska,319777.0,321652.0,327054.0,332494.0,337469.0,344157.0,348756.0,350233.0,...,6.7,6.3,6.6,6.5,6.0,5.6,8.3,6.4,4.0,NaN
1,AL,Alabama,2147173.0,2128027.0,2112621.0,2128668.0,2138306.0,2140356.0,2170007.0,2180448.0,...,6.7,6.1,5.9,4.5,3.9,3.2,6.4,3.4,2.6,NaN
2,AR,Arkansas,1260507.0,1261419.0,1276964.0,1282578.0,1302378.0,1333880.0,1346532.0,1350811.0,...,5.9,5.0,4.0,3.7,3.7,3.5,6.2,4.1,3.3,NaN
3,AZ,Arizona,2510611.0,2593664.0,2683502.0,2728649.0,2788618.0,2862517.0,2970679.0,3018805.0,...,6.8,6.1,5.5,5.0,4.8,4.8,7.8,5.1,3.8,NaN
4,CA,California,16837536.0,17096428.0,17246528.0,17271899.0,17374506.0,17537931.0,17661179.0,17910726.0,...,7.6,6.3,5.5,4.8,4.2,4.1,10.1,7.3,4.2,NaN


In [20]:
# Save the pivoted dataset to a CSV file
output_csv_path = 'USDA_dataset_1.csv'
dataset_pivoted.to_csv(output_csv_path, index=False)
print(f"Dataset successfully saved to {output_csv_path}")

# Verify the file was created
import os
if os.path.exists(output_csv_path):
    print(f"File size: {os.path.getsize(output_csv_path) / (1024*1024):.2f} MB")
    # Display the first few rows of the saved file to verify
    df_verification = pd.read_csv(output_csv_path, nrows=5)
    print("\nPreview of saved CSV file:")
    print(df_verification.head())
else:
    print("Error: File was not created successfully")


Dataset successfully saved to USDA_dataset_1.csv
File size: 0.04 MB

Preview of saved CSV file:
  State State (long)  Civilian labor force (2000)  \
0    AK       Alaska                     319777.0   
1    AL      Alabama                    2147173.0   
2    AR     Arkansas                    1260507.0   
3    AZ      Arizona                    2510611.0   
4    CA   California                   16837536.0   

   Civilian labor force (2001)  Civilian labor force (2002)  \
0                     321652.0                     327054.0   
1                    2128027.0                    2112621.0   
2                    1261419.0                    1276964.0   
3                    2593664.0                    2683502.0   
4                   17096428.0                   17246528.0   

   Civilian labor force (2003)  Civilian labor force (2004)  \
0                     332494.0                     337469.0   
1                    2128668.0                    2138306.0   
2                